# Modelado Dimensional: Esquema Estrella y Copo de Nieve

Objetivo: Entender hechos y dimensiones, diferencias entre estrella y copo de nieve, cuándo desnormalizar y optimizar para analítica.

Contenido compacto para facilitar lectura (secciones cortas + ejemplos).

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Modelos Conceptual, Lógico y Físico: Los Tres Esquemas](04_modelado_tres_esquemas.ipynb) | Siguiente: [Fundamentos de Calidad de Datos (Junior)](06_fundamentos_calidad_datos.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. 1. Fundamentos Rápidos
2. 2. Hechos vs Dimensiones
3. 3. Esquema Estrella (Star Schema)
4. 4. Esquema Copo de Nieve (Snowflake)
5. 5. Normalización vs Desnormalización
6. 6. Optimización Analítica
7. 7. Ejercicio Guiado
8. 8. Checklist Rápido para tu Diseño

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Modelado Dimensional: Esquema Estrella y Copo de Nieve"] --> C["Notebook siguiente"]
```


## 1. Fundamentos Rápidos

Modelado dimensional ≠ modelado relacional tradicional. Se orienta a consulta analítica (OLAP): agregaciones, filtros, drill-down.

Estructura base:
- Tabla de hechos (hechos cuantitativos)
- Tablas de dimensiones (contexto descriptivo)

Principios clave (Kimball):
1. Entender el proceso de negocio (evento medible)
2. Declarar la granularidad (nivel mínimo de detalle)
3. Identificar dimensiones (describen el evento)
4. Identificar hechos (medidas numéricas aditivas)

## 2. Hechos vs Dimensiones

Hecho: Registro medible de un evento (ej: venta).
Dimensión: Punto de vista para filtrar/segmentar (ej: producto, cliente, fecha).

Tipos de hechos:
- Transaccionales (cada evento)
- Acumulados (snapshot)
- Periodos (cambios en intervalos)

Tipos de atributos dimensionales:
- Descriptivos (nombre_producto)
- Jerárquicos (día → mes → año)
- Derivados (segmento_cliente)
- BW Flags (es_promocion)

In [ ]:
# Simulación simple de un proceso de ventas para ilustrar hecho y dimensiones
import pandas as pd
import numpy as np

np.random.seed(42)

dim_producto = pd.DataFrame({
    'producto_sk': [101, 102, 103],
    'producto_id_fuente': ['P-1', 'P-2', 'P-3'],
    'nombre_producto': ['Laptop', 'Mouse', 'Monitor'],
    'categoria': ['Computo', 'Accesorios', 'Computo'],
    'subcategoria': ['Portátiles', 'Periféricos', 'Pantallas']
})

dim_cliente = pd.DataFrame({
    'cliente_sk': [201, 202, 203],
    'cliente_id_fuente': ['C-1', 'C-2', 'C-3'],
    'nombre_cliente': ['Ana', 'Luis', 'María'],
    'segmento': ['Premium', 'Estándar', 'Premium'],
    'pais': ['CO', 'CO', 'MX']
})

dim_tiempo = pd.DataFrame({'fecha_sk': pd.date_range('2025-01-01', '2025-01-05')})
dim_tiempo['dia'] = dim_tiempo['fecha_sk'].dt.day
dim_tiempo['mes'] = dim_tiempo['fecha_sk'].dt.month
dim_tiempo['anio'] = dim_tiempo['fecha_sk'].dt.year
dim_tiempo['mes_texto'] = dim_tiempo['fecha_sk'].dt.month.map({1: 'Ene'})

hechos_ventas = []
for fecha in dim_tiempo['fecha_sk']:
    for producto_sk in dim_producto['producto_sk']:
        for cliente_sk in dim_cliente['cliente_sk']:
            cantidad = np.random.poisson(1.2)
            if cantidad > 0:
                precio = {101: 900, 102: 25, 103: 300}[producto_sk]
                hechos_ventas.append([fecha, producto_sk, cliente_sk, cantidad, cantidad * precio])

fact_ventas = pd.DataFrame(
    hechos_ventas,
    columns=['fecha_sk', 'producto_sk', 'cliente_sk', 'cantidad', 'importe']
)

print('Dim Producto:')
print(dim_producto)
print('\nDim Cliente:')
print(dim_cliente)
print('\nDim Tiempo (extracto):')
print(dim_tiempo.head(3))
print('\nFact Ventas (primeras filas):')
print(fact_ventas.head())
print('\nGranularidad definida: una fila = (fecha, producto, cliente)')


## 3. Esquema Estrella (Star Schema)

Características:
- Dimensiones desnormalizadas (atributos completos)
- Joins simples (fact ↔ dimensión por surrogate key)
- Rendimiento alto en consultas analíticas
- Facilidad de comprensión para usuarios de negocio

Desventaja: Puede haber redundancia en dimensiones amplias (ej: país → región → continente en misma tabla).

## 4. Esquema Copo de Nieve (Snowflake)

Características:
- Dimensiones normalizadas en múltiples tablas
- Menos redundancia, más joins
- Puede reducir almacenamiento y facilitar mantenimiento jerárquico

Desventaja: Consultas más complejas y potencial menor rendimiento.

Uso típico: Dimensiones con jerarquías extensas y relativamente estáticas (geografía, producto complejo).

In [ ]:
# Transformar dim_producto a estructura snowflake (separar categoria y subcategoria)
dim_categoria = dim_producto[['categoria']].drop_duplicates().reset_index(drop=True)
dim_categoria['categoria_sk'] = dim_categoria.index + 500

dim_subcat = (
    dim_producto[['subcategoria', 'categoria']]
    .drop_duplicates()
    .merge(dim_categoria, on='categoria')
    .reset_index(drop=True)
)
dim_subcat['subcategoria_sk'] = dim_subcat.index + 600

dim_producto_snow = (
    dim_producto
    .merge(dim_subcat[['subcategoria', 'subcategoria_sk']], on='subcategoria')
    .merge(dim_categoria[['categoria', 'categoria_sk']], on='categoria')
    [['producto_sk', 'nombre_producto', 'subcategoria_sk', 'categoria_sk']]
)

print('Dim Producto (estrella):')
print(dim_producto)
print('\nDim Producto (snowflake):')
print(dim_producto_snow)
print('\nDim Subcategoria:')
print(dim_subcat)
print('\nDim Categoria:')
print(dim_categoria)


In [ ]:
# Consulta simulada: importe total por categoria vs tiempo (estrella vs copo)
import time

t0 = time.time()
q_estrella = (
    fact_ventas
    .merge(dim_producto[['producto_sk', 'categoria']], on='producto_sk')
    .groupby('categoria', as_index=False)['importe'].sum()
)
t1 = time.time() - t0

t2 = time.time()
q_copo = (
    fact_ventas
    .merge(dim_producto_snow, on='producto_sk')
    .merge(dim_categoria[['categoria', 'categoria_sk']], on='categoria_sk')
    .groupby('categoria', as_index=False)['importe'].sum()
)
t3 = time.time() - t2

print('Resultado Estrella:')
print(q_estrella)
print('Tiempo Estrella (ms):', round(t1 * 1000, 2))
print('\nResultado Copo de Nieve:')
print(q_copo)
print('Tiempo Copo (ms):', round(t3 * 1000, 2))
print('\nNota: la diferencia es mínima porque el dataset es pequeño.')


## 5. Normalización vs Desnormalización

| Aspecto | Normalizado (Snowflake) | Desnormalizado (Estrella) |
|---------|-------------------------|---------------------------|
| Redundancia | Baja | Moderada |
| Rendimiento analítico | Menor (más joins) | Mayor (joins simples) |
| Facilidad de uso | Menor (estructura fragmentada) | Alta (intuitivo) |
| Mantenimiento jerarquías | Más fácil | Puede implicar updates masivos |
| Almacenamiento | Menor | Mayor |

Criterios para elegir desnormalizar:
- Frecuencia alta de consultas agregadas
- Usuarios no técnicos consumen el modelo
- Minimizar tiempos de respuesta en dashboards

Criterios para mantener normalización parcial:
- Dimensiones muy grandes (geografía global detallada)
- Jerarquías complejas con cambios frecuentes
- Optimización de almacenamiento en data warehouse masivo

## 6. Optimización Analítica

Técnicas clave:
1. Surrogate Keys (enteros secuenciales) → Joins más rápidos
2. Columnar Storage (en data warehouse moderno) → Lectura selectiva
3. Particionamiento por fecha o región → Pruning
4. Índices bitmap (algunos motores) → Filtros rápidos en baja cardinalidad
5. Pre-Agregaciones / Tablas resumen → Menos cómputo en tiempo real
6. Caché en motores MPP (BigQuery, Redshift) → Reutilización plan/resultado
7. Ordenar físicamente por surrogate keys y fecha para compresión y scans eficientes.

Evitar: Desnormalizar exageradamente creando dimensiones gigantes de 1..N atributos no usados; sobre-indexar tablas de hechos.

## 7. Ejercicio Guiado

Escenario: Marketplace con ventas multi-país y campañas de marketing.

Tareas:
1. Define granularidad para fact_ventas_marketing (¿por evento de clic? ¿por transacción?)
2. Lista dimensiones necesarias (producto, cliente, fecha, campaña, canal, país).
3. Clasifica cuáles podrían normalizarse en snowflake (ej: país → región → continente).
4. Especifica tres métricas (hechos) aditivas.
5. Propón un resumen mensual pre-agregado.

Completa en la celda siguiente.

In [ ]:
# Plantilla de respuesta del ejercicio
plantilla = '''
# Ejercicio: Diseño Modelo Dimensional Marketplace

1. Granularidad: [Describe aquí]
2. Dimensiones: [Lista]
3. Jerarquías a snowflake: [Lista]
4. Métricas aditivas: [Lista y justificación de aditividad]
5. Tabla resumen mensual: [Columnas y propósito]

Riesgos si se desnormaliza demasiado: [Explica]
Compensaciones rendimiento vs mantenimiento: [Explica]
'''
print(plantilla)

## 8. Checklist Rápido para tu Diseño

- [ ] Granularidad clara y documentada
- [ ] Hechos son numéricos y aditivos (o semi-aditivos documentados)
- [ ] Dimensiones contienen atributos útiles de filtrado
- [ ] Claves sustitutas (surrogate) implementadas
- [ ] Jerarquías definidas (fecha, producto, geografía)
- [ ] Estrategia de slowly changing dimensions (SCD) considerada (se verá más adelante)
- [ ] Plan de índices / particiones alineado a patrones de consulta
- [ ] Justificación de cualquier desnormalización
- [ ] Tabla(s) de agregación si consultas repetitivas pesadas

## 9. Resumen

- Hechos registran eventos y contienen métricas.
- Dimensiones describen el contexto y soportan cortes analíticos.
- Estrella: simplicidad y velocidad; Copo: ahorro y estructura jerárquica.
- Desnormalizar es una decisión consciente para acelerar analítica.
- Optimizar implica balancear: rendimiento, claridad, mantenimiento.

Siguiente tema (futuro): Slowly Changing Dimensions (SCD) y manejo de historiales.

---
**Referencias Breves**
- Kimball, R. *The Data Warehouse Toolkit*.
- Inmon, W. *Building the Data Warehouse*.
- Corr, L. *Agile Data Warehouse Design*.
- Gupta, A. *Database System Concepts* (capítulo data warehousing).

## 16. Actualización 2025: Modelado Dimensional Avanzado

### Métricas Modernas y Consistencia
La presión por métricas gobernadas impulsa el uso de capas semánticas (dbt Semantic Layer, Cube, Transform) para centralizar definiciones. Evita duplicar lógica de cálculo en múltiples dashboards.

### SCD (Slowly Changing Dimensions) Relevantes
| Tipo | Uso | Ventajas | Consideraciones |
|------|-----|----------|-----------------|
| Tipo 1 | Corrección sin historial | Simplicidad | Pérdida de historia |
| Tipo 2 | Historial completo (vigencia) | Auditoría, evolución analítica | Más filas, manejo fechas |
| Tipo 3 | Escenarios "antes vs ahora" limitados | Comparaciones puntuales | Mantiene pocas versiones |

En 2025 la combinación Tipo 1 + Tipo 2 cubre >90% de necesidades. Tipo 3 aparece en casos muy específicos.

### Buenas Prácticas SCD Tipo 2
- Usar columnas: `valid_from`, `valid_to` (`NULL` o fecha futura) y flag `is_current`.
- Garantizar no solapamiento de intervalos por `customer_id` (regla de calidad).
- Implementar merges idempotentes (Spark Merge, dbt incremental) apoyados en hashes de atributos.
- Tests automáticos: unicidad combinación (`customer_id`,`valid_from`), no superposición intervalos, exactamente 1 registro `is_current`.

### Métricas Derivadas (Ejemplos Sin Alucinación)
| Métrica | Definición | Dimensión | Grano |
|---------|-----------|----------|-------|
| customer_lifetime_days | Días entre primer y último evento compra | Customer (SCD2) | Customer |
| active_customers_month | Count clientes con compra en mes | Calendar | Mes |
| avg_revenue_per_customer | Ingreso / clientes activos | Customer+Fact Sales | Mes |
| churn_rate | (customers_inactivos / clientes_previos) | Customer SCD2 | Mes |
| new_vs_returning_ratio | Nuevos / Returning | Customer SCD2 | Mes |

## 17. Mini-Ejercicio: Implementación Lógica SCD Tipo 2

### Escenario
Tienes una dimensión cliente base (`dim_customer_current`) y un flujo de cambios diarios (`customer_changes`) que incluye modificaciones de segmento y país.
Objetivo: Generar una tabla SCD2 (`dim_customer_scd2`) aplicando nuevas versiones sin perder histórico.

### Estructuras (ejemplo simplificado)
- `dim_customer_current(customer_id, name, segment, country, updated_at)`
- `customer_changes(customer_id, segment, country, change_ts)`

### Tareas
1. Definir columnas finales de `dim_customer_scd2` (mínimo: customer_id, name, segment, country, valid_from, valid_to, is_current).
2. Describir algoritmo (pasos) para procesar cada batch de cambios (insertar nuevas versiones y cerrar anteriores).
3. Indicar 5 reglas de calidad: no solapamiento intervalos, una fila vigente, valores no nulos críticos, fechas consistentes (`valid_from < valid_to`), país válido.
4. Escribir pseudocódigo (o Python base) que genere versiones a partir de DataFrames simulados.
5. Proponer 3 métricas de monitoreo: porcentaje registros con intervalos válidos, tiempo de ejecución merge, recuento de versiones por cliente (outliers > X).

### Rúbrica
| Aspecto | Peso |
|---------|------|
| Definición estructura SCD2 | 25% |
| Algoritmo pasos claro | 25% |
| Reglas calidad medibles | 20% |
| Pseudocódigo correcto | 20% |
| Métricas monitoreo | 10% |

### Siguientes Pasos (Opcional)
- Implementar en dbt: modelo incremental + tests (unique, relationships, custom no-overlap).
- Añadir validación Great Expectations para no solapamiento (custom expectation).
- Versionar contrato de dimensión cliente.

> Este ejercicio conecta con notebooks de gobernanza y calidad para asegurar histórico confiable.


In [ ]:
# Pseudocódigo Python base para SCD Tipo 2 (estudiante puede expandir)
import pandas as pd

# DataFrames simulados (placeholder)
dim_current = pd.DataFrame([
    {"customer_id": 1, "name": "Alice", "segment": "SMB", "country": "US", "updated_at": "2025-11-01"},
    {"customer_id": 2, "name": "Bob", "segment": "Enterprise", "country": "DE", "updated_at": "2025-11-01"},
])
changes = pd.DataFrame([
    {"customer_id": 1, "segment": "Enterprise", "country": "US", "change_ts": "2025-11-15"},
    {"customer_id": 2, "segment": "Enterprise", "country": "FR", "change_ts": "2025-11-16"},
])

# Normalizar fechas
dim_current['updated_at'] = pd.to_datetime(dim_current['updated_at'])
changes['change_ts'] = pd.to_datetime(changes['change_ts'])

# Tabla SCD2 inicial (asumimos una fila vigente por cliente)
scd2 = dim_current.copy()
scd2['valid_from'] = scd2['updated_at']
scd2['valid_to'] = pd.NaT
scd2['is_current'] = True
scd2 = scd2.drop(columns=['updated_at'])

# Procesar cambios
for _, ch in changes.sort_values('change_ts').iterrows():
    cid = ch['customer_id']
    # Cerrar registro vigente
    mask_current = (scd2['customer_id'] == cid) & (scd2['is_current'])
    if mask_current.any():
        scd2.loc[mask_current, 'valid_to'] = ch['change_ts']
        scd2.loc[mask_current, 'is_current'] = False
    # Insertar nueva versión
    new_row = {
        'customer_id': cid,
        'name': scd2.loc[mask_current, 'name'].iloc[0] if mask_current.any() else None,
        'segment': ch['segment'],
        'country': ch['country'],
        'valid_from': ch['change_ts'],
        'valid_to': pd.NaT,
        'is_current': True
    }
    scd2 = pd.concat([scd2, pd.DataFrame([new_row])], ignore_index=True)

print("SCD2 result:")
print(scd2.sort_values(['customer_id','valid_from']))

# Reglas de calidad rápidas
# 1. No solapamiento: cada customer_id debe tener intervalos consecutivos sin overlap
violations_overlap = 0
for cid, grp in scd2.groupby('customer_id'):
    grp_sorted = grp.sort_values('valid_from')
    prev_end = None
    for _, row in grp_sorted.iterrows():
        if prev_end and pd.notna(prev_end) and row['valid_from'] < prev_end:
            violations_overlap += 1
        prev_end = row['valid_to']
print("Violaciones solapamiento:", violations_overlap)

# 2. Una fila vigente
multiple_current = scd2.groupby('customer_id')['is_current'].sum().gt(1).sum()
print("Clientes con múltiples filas vigentes:", multiple_current)


<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- Referencias del curso: ../04-recursos/referencias.md
